In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle'):
    print(dirname)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# !pip install pyarabic

In [ ]:
# %env JOBLIB_TEMP_FOLDER=/tmp

In [ ]:
from tqdm import tqdm
import unicodedata
import re
import zipfile
import os
from pyarabic.araby import tokenize, is_arabicrange, strip_tashkeel, normalize_hamza,\
                           strip_tatweel, strip_lastharaka
from pyarabic.trans import normalize_digits

In [ ]:
def digit_convertor(text):
    return normalize_digits(text, source='all', out='east')
def tasheel_hamza(text):
    return normalize_hamza(text, method="tasheel")

def arabic_normlizer(text):
    text = tasheel_hamza(text)
    text = digit_convertor(text)
#     text = strip_tashkeel(text)
    text = strip_tatweel(text)
    text = strip_lastharaka(text)
    return text

In [ ]:
def unicode_to_ascii(s):
    '''
    Args:
        s : UniCode file
    Returns:
        ASCII file
    
    Converts the unicode file to ascii
    For each character:
                        there are two normal forms: normal form C and normal form D. 
                                                    - Normal form D (NFD) is also known as canonical decomposition,
                                                      and translates each character into its decomposed form. 
                                                      
                                                    - Normal form C (NFC) first applies a canonical decomposition, 
                                                    then composes pre-combined characters again.
    '''
    return ''.join(c for c in unicodedata.normalize('NFD', s)) # if unicodedata.category(c) != 'Mn'# Mn ==> Mark, Nonspacing (التشكيل )


In [ ]:
def preprocess_sentence(s, i=0):
    '''
    Args:
        s : A single sentence 
    Returns:
        s : Single normalize sentence 
    
    Convert Unicode to ASCII
    Creating a space between a word and the punctuation following it
    eg: "he is a boy." => "he is a boy ." 
    Reference:- https://stackoverflow.com/questions/3645931/python-padding-punctuation-with-white-spaces-keeping-punctuation
    
    Replacing everything with space except (a-z, A-Z, ا-ي ".", "?", "!", ",")
    
    Adding a start and an end token to the sentence
    
    '''
    if i:
        s = arabic_normlizer(s)
        s = unicode_to_ascii(s.strip())
    
#         s = re.sub(r"([?.!,¿؟])", r" \1 ", s)
        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^أ-ۿ١-٩اءئ؟!.',?_،إ]+", " ", s)
    else:
#         s = unicode_to_ascii(s.lower().strip())
    
#         s = re.sub(r"([?.!,¿])", r" \1 ", s)
        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^a-zA-Z0-9?.'!,¿]+", " ", s)
    
    s = s.rstrip().strip()
#     s = f'<start> {s} <end>'
    return s


In [ ]:
t = '''فَرَاشَةٌ مُلَوَّنَةٌ تَطِيْرُ في البُسْتَانِ، حُلْوَةٌ مُهَنْدَمَةٌ تُدْهِشُ الإِنْسَانَ، أَهْدَافُهَا مُحَدَّدَةٌ، حَرَكَاتُها مُرَتَّبَةٌ، تَحُوْمُ بِانْتِظَامٍ، تَحُطُّ فِي نُعُومَةٍ تَنْشُرُ السَّلاَمَ. فَرَاشَةٌ مُلَوَّنَةٌ تَطِيرُ بِلا اُنْقِطَاعٍ، بِالنَّهارِ المُشْرِقِ تَمْلَأُ البِقَاعَ، تُحِبُّ الوَرْدَ المَزْرُوعَ، تَلْثُمُهُ فِي وَقْتِ الجُوعِ، تَمْتَصُّ رَحِيْقَ الأَزْهَارِ، تُحْيي جَنْيَ الأَشْجَارِ، مِنْ وَرْدَةٍ لِوَرْدَةٍ، تَطِيْرُ بِانْتِظَامٍ.

فَرَاشَةٌ مُلَوَّنَةٌ تُعْطِي بِلا انْتِفَاعٍ، هَمُّها ثِمَارٌ تُطْعِمُ الجِيَاعَ، تُحَاكي الجَمَالَ، تَرْسُمُ الإِبْدَاعَ.. تَعِيشُ في وِئَامٍ. صَحِيْحٌ أَنَّها ضَعِيفَةٌ.. ضَعِيفَةٌ، لَيْسَ لَهَا مِنْ قُوَّة، وَلاَ لَهَا مِنْ حِيْلَة، فَسُبْحَانَ مَنْ أَعْطَاها أَلْوَانَها الفَرِيْدَةَ وَمَزَايَاهَا العَدِيْدَة، فَدَوْرُها كبيرٌ، بِحَجْمِها الصَّغِيْرِ، سُبْحَانَ مَنْ أَعْطَاهَا فَوَائِدَها العَظِيمَةَ، فَهَلاَّ تَعَلَّمْنَا خِصَالَها الحَمِيْدَةَ، وَكُنّا مِثْلَها نَعِيشُ فِي سَلامٍ.. وانْتِظامٍ… وَوِئَامٍ، حياتُنَا رغيدةٌ، خَيْرَاتُنَا أكيدَةٌ… وصَمْتُنَا نَمَاءٌ… وصَمْتُنَا عَطَاءٌ… وصَمْتُنَا كلاَمٌ.
'''

x = ''' Technical and professional education shall be made generally available and higher education shall be equally accessible to all on the basis of merit.
2. Education shall be directed to the full development of the human personality and to the strengthening of respect for human rights and fundamental freedoms. It shall promote understanding, tolerance and friendship among all nations, racial or religious groups, and shall further the activities of the United Nations for the maintenance of peace.
3. Parents have a prior right to choose the kind of education that shall be given to their children.
Article 27
1. Everyone has the right freely to participate in the cultural life of the community, to enjoy the arts and to share in scientific advancement and its benefits.
2. Everyone has the right to the protection of the moral and material interests resulting from any scientific, literary or artistic production of which he is the author.
'''
print(preprocess_sentence(t, 1))
print(preprocess_sentence(x, 0))

In [ ]:
def sort_freq(pair):
            return pair[1]
    
class GetVoc:
    def __init__(self):
        self.voc = dict()
        self.last_id = 0
        self.freq = dict()
            
    def extend_from_text(self, text, frq_threshold=None):
        text = text.split()
        self._extend_freq(text)
        
        if frq_threshold is None:
            text = set(text)
            for i in text:
                if not self.voc.get(i, 0):
                    self.last_id += 1
                    self.voc[i] = self.last_id
        else:
            for i in sorted(self.freq.items(), key=sort_freq, reverse=True):
                if i[1] >= frq_threshold:
                    if not self.voc.get(i[0], 0):
                        self.last_id += 1
                        self.voc[i[0]] = self.last_id
                
    def extend(self, other):
        for i in other.keys():
            if not self.voc.get(i, 0):
                self.last_id += 1
                self.voc[i] = self.last_id
    
    def _extend_freq(self, text):
        for word in text:
            if self.freq.get(word, 0):
                self.freq[word] += 1
            else:
                self.freq[word] = 1
       
    def save_voc(self, path, file_name):
        with open(f'{path}/{file_name}.txt', 'w', encoding='utf-8-sig') as f:
            for key, val in self.voc.items():
                f.write(f'{key}:{val}\n')
                
        with open(f'{path}/{file_name}_freq.txt', 'w', encoding='utf-8-sig') as f:
            for key, val in self.freq.items():
                f.write(f'{key}:{val}\n')

In [ ]:
def save_data(archive_path, data_dir):
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(f'{data_dir}/vocublary', exist_ok=True)
    os.makedirs(f'{data_dir}/ar', exist_ok=True)
    os.makedirs(f'{data_dir}/en', exist_ok=True)
    ar_file = 'OpenSubtitles.ar-en.ar'
    en_file = 'OpenSubtitles.ar-en.en'
    with zipfile.ZipFile(archive_path, mode="r") as archive:
        
        with archive.open(ar_file, mode="r") as f:
            ar_voc = GetVoc()
            state = 1
            two_sent = ''
            k = 1 
            batch_sent = []
            for text in tqdm(f):
                text = preprocess_sentence(text.decode(), 1)
                ar_voc.extend_from_text(text)
                two_sent += text
                if not state % 3:
                    two_sent += '\n'
                    
                    batch_sent.append(two_sent)
                    two_sent = ''
                    
                state += 1
                
                if not state % 20000:
#                     print(len(batch_sent))
                    with open(f'{data_dir}/ar/batch_{k}.txt', 'w', encoding='utf-8-sig') as f:
                        f.writelines(batch_sent)
                        k += 1
                        batch_sent.clear()
                    
            ar_voc.save_voc(f'{data_dir}/vocublary', 'ar')
            print(len(ar_voc.voc))
            
        with archive.open(en_file, mode="r") as f:
            en_voc = GetVoc()
            state = 0
            two_sent = ''
            k = 1 
            batch_sent = []
            for text in tqdm(f):
                text = preprocess_sentence(text.decode())
                en_voc.extend_from_text(text)
                two_sent += text
                if not state % 3 and state:
                    two_sent += '\n'
                    batch_sent.append(two_sent)
                    two_sent = ''
                    
                state += 1
                
                if not state % 20000:
                    with open(f'{data_dir}/en/batch_{k}.txt', 'w', encoding='utf-8-sig') as f:
                        f.writelines(batch_sent)
                        k += 1
                        batch_sent.clear()
                    
            en_voc.save_voc(f'{data_dir}/vocublary', 'en')
            print(len(en_voc.voc))
            

In [ ]:
DATA_PATH = '/kaggle/input/en-ar-zip-data/download.php?f=OpenSubtitles%2Fv2018%2Fmoses%2Far-en.txt.zip'

save_data(DATA_PATH, '/kaggle/working/')